# BirdCLEF 2026: Data Refinement EDA
## Visualizing Fusion-Guided Peak Selection & Contextual Stitching

This notebook analyzes the results of the **Phase 2: Soundscape Data Refinement** strategy. We compare our previous baseline (taking the first 5 seconds of every recording) with our new "Unified 5s" dataset which uses Model-Guided and Energy-Based logic to center bird calls.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
from src.audio.spectrograms import SpectrogramGenerator

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Load Manifests
We load both the intermediate peak manifest and the final master manifest.

In [ ]:
peaks_df = pd.read_csv('../data/processed/train_v2_peaks_fast.csv')
master_df = pd.read_csv('../data/processed/train_v2_master_fast.csv')
train_df = pd.read_csv('../data/raw/train.csv')

print(f"Total Samples: {len(peaks_df)}")
peaks_df.head()

## 2. Global Statistics
### Selection Method Distribution
How often did the model find a confident peak vs. falling back to RMS energy?

In [ ]:
plt.figure(figsize=(10, 6))
method_counts = peaks_df['method'].value_counts(normalize=True) * 100
sns.barplot(x=method_counts.index, y=method_counts.values, palette="viridis")
plt.title("Peak Selection Method Distribution")
plt.ylabel("Percentage of Dataset")
plt.show()

### The "Shift" Analysis
How many bird calls actually started after the first 5 seconds?

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(peaks_df[peaks_df['start_time'] > 0]['start_time'], bins=50, kde=True, color="orange")
plt.title("Distribution of Peak Start Times (Excluding 0-5s Start)")
plt.xlabel("Start Time (seconds)")
plt.ylabel("Frequency")
plt.show()

shifted_pct = (peaks_df['start_time'] > 0).mean() * 100
print(f"Percentage of samples shifted away from baseline: {shifted_pct:.1f}%")

## 3. Confidence Analysis
Exploring how confident the Fusion V1 model was during the scan.

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(peaks_df[peaks_df['method'] == 'fusion']['confidence'], fill=True, color="green", label="Fusion Guided")
plt.title("Confidence Score Distribution (Fusion Method)")
plt.xlabel("Softmax Confidence")
plt.show()

## 4. Visual Verification (Before vs After)
Let's compare the spectrogram of the first 5s (Old Baseline) with the selected Peak 5s (New Unified).

In [ ]:
def compare_peaks(filename, start_time):
    audio_path = os.path.join('../data/raw/train_audio', filename)
    y, sr = librosa.load(audio_path, sr=32000, duration=max(10, start_time + 5))
    
    # Extract Baseline (0-5s)
    y_old = y[0:sr*5]
    # Extract Peak (start_time to start_time+5)
    s_idx = int(start_time * sr)
    y_new = y[s_idx : s_idx + sr*5]
    
    fig, ax = plt.subplots(2, 1, figsize=(15, 10))
    
    S_old = librosa.feature.melspectrogram(y=y_old, sr=sr)
    librosa.display.specshow(librosa.power_to_db(S_old, ref=np.max), ax=ax[0], x_axis='time', y_axis='mel')
    ax[0].set_title(f"Baseline (0-5s): {filename}")
    
    S_new = librosa.feature.melspectrogram(y=y_new, sr=sr)
    librosa.display.specshow(librosa.power_to_db(S_new, ref=np.max), ax=ax[1], x_axis='time', y_axis='mel')
    ax[1].set_title(f"Selected Peak ({start_time}s - {start_time+5}s)")
    
    plt.tight_layout()
    plt.show()

# Pick a highly shifted sample with high confidence
sample = peaks_df[(peaks_df['start_time'] > 10) & (peaks_df['confidence'] > 0.9)].iloc[0]
compare_peaks(sample['filename'], sample['start_time'])

## 5. Stitching Verification
Looking at how short clips were padded with noise.

In [ ]:
stitched_samples = master_df[master_df['method'] == 'stitched'].head(3)

for idx, row in stitched_samples.iterrows():
    path = os.path.join('../data/processed/train_v2_unified', row['unified_filename'])
    if os.path.exists(path):
        y, sr = librosa.load(path, sr=32000)
        plt.figure(figsize=(15, 3))
        librosa.display.waveshow(y, sr=sr, color="blue")
        plt.title(f"Stitched Sample: {row['unified_filename']} (Centered)")
        plt.show()
        # ipd.display(ipd.Audio(y, rate=sr)) # Uncomment to listen